In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.retail.bronze_sales")
print("Bronze rows:", bronze.count())

# 1. Remove exact duplicate orders (keep one row per order_id)
deduped = bronze.dropDuplicates(["order_id"])

# 2. Fix types and formats
#    try_* functions return NULL instead of failing on bad values
cleaned = (
    deduped
    .withColumn(
        "order_date",
        F.coalesce(
            F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
            F.expr("try_to_timestamp(order_date, 'dd/MM/yyyy')")
        ).cast("date")
    )
    .withColumn("quantity", F.expr("try_cast(quantity as int)"))
    .withColumn("unit_price", F.expr("try_cast(unit_price as double)"))
    .withColumn("region", F.initcap(F.trim(F.col("region"))))
    .withColumn("total_amount", F.col("quantity") * F.col("unit_price"))
    .withColumn("processed_at", F.current_timestamp())
)

# 3. Split valid and invalid rows
is_valid = (
    F.col("customer_id").isNotNull()
    & F.col("quantity").isNotNull()
    & (F.col("quantity") > 0)
    & F.col("order_date").isNotNull()
)

silver = cleaned.filter(is_valid)
rejected = cleaned.filter(~is_valid)

# 4. Write Delta tables
silver.write.mode("overwrite").saveAsTable("workspace.retail.silver_sales")
rejected.write.mode("overwrite").saveAsTable("workspace.retail.silver_sales_rejected")

print("Silver rows:", silver.count())
print("Rejected rows:", rejected.count())

Bronze rows: 2050
Silver rows: 1859
Rejected rows: 141


In [0]:
display(spark.table("workspace.retail.silver_sales").limit(10))
display(spark.table("workspace.retail.silver_sales_rejected").limit(10))

order_id,order_date,customer_id,product_id,product_name,quantity,unit_price,region,total_amount,processed_at
O01383,2026-04-11,C0150,P006,Mouse,3,25.0,North,75.0,2026-11-11T13:40:33.487Z
O01396,2026-05-27,C0177,P006,Mouse,3,25.0,South,75.0,2026-11-11T13:40:33.487Z
O01408,2026-04-18,C0161,P002,Phone,2,600.0,West,1200.0,2026-11-11T13:40:33.487Z
O01411,2026-05-17,C0099,P006,Mouse,4,25.0,East,100.0,2026-11-11T13:40:33.487Z
O01435,2026-10-13,C0289,P005,Keyboard,1,45.0,West,45.0,2026-11-11T13:40:33.487Z
O01456,2026-06-19,C0040,P005,Keyboard,2,45.0,South,90.0,2026-11-11T13:40:33.487Z
O01467,2026-07-22,C0201,P003,Headphones,3,80.0,West,240.0,2026-11-11T13:40:33.487Z
O01485,2026-01-02,C0223,P001,Laptop,3,900.0,West,2700.0,2026-11-11T13:40:33.487Z
O01516,2026-06-19,C0270,P006,Mouse,3,25.0,South,75.0,2026-11-11T13:40:33.487Z
O01518,2026-01-27,C0132,P004,Monitor,3,250.0,North,750.0,2026-11-11T13:40:33.487Z


order_id,order_date,customer_id,product_id,product_name,quantity,unit_price,region,total_amount,processed_at
O01575,2026-09-02,C0083,P003,Headphones,null,80.0,West,null,2026-11-11T13:40:38.004Z
O01623,2026-10-28,C0058,P001,Laptop,null,900.0,South,null,2026-11-11T13:40:38.004Z
O01843,2026-04-13,C0037,P002,Phone,null,600.0,West,null,2026-11-11T13:40:38.004Z
O01098,2026-06-05,null,P004,Monitor,2,250.0,North,500.0,2026-11-11T13:40:38.004Z
O01578,2026-10-12,C0272,P002,Phone,null,600.0,South,null,2026-11-11T13:40:38.004Z
O01693,2026-09-26,null,P006,Mouse,1,25.0,West,25.0,2026-11-11T13:40:38.004Z
O01868,2026-06-07,null,P003,Headphones,3,80.0,North,240.0,2026-11-11T13:40:38.004Z
O01381,2026-07-05,C0293,P005,Keyboard,-1,45.0,South,-45.0,2026-11-11T13:40:38.004Z
O01407,2026-09-29,C0066,P002,Phone,null,600.0,East,null,2026-11-11T13:40:38.004Z
O01512,2026-04-11,C0142,P002,Phone,null,600.0,West,null,2026-11-11T13:40:38.004Z


In [0]:
%sql
DESCRIBE HISTORY workspace.retail.silver_sales;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-11T13:40:36.000Z,73729152433193,anupriya.singh145@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""5b635da1-488e-4cf4-b614-6f622396b902"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-2073c408-9e87-4264-ba39-a9ea4e9e44f7"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-ab838920-dbd7-40d3-892e-cd258e0ae71a""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(695014744698147),65887895-08cc-43e3-916a-47139f217a8d,1111-133454-rb8bgmy7-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1859, numOutputBytes -> 17699)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT COUNT(*) FROM workspace.retail.silver_sales VERSION AS OF 0;

COUNT(*)
1859


In [0]:
%sql
DELETE FROM workspace.retail.silver_sales WHERE region = 'North';

num_affected_rows
461


In [0]:
%sql
SELECT COUNT(*) AS current_rows FROM workspace.retail.silver_sales;

current_rows
1398


In [0]:
%sql
SELECT COUNT(*) AS rows_before_delete FROM workspace.retail.silver_sales VERSION AS OF 0;

rows_before_delete
1859


In [0]:
%sql
DESCRIBE HISTORY workspace.retail.silver_sales;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-11T13:43:19.000Z,73729152433193,anupriya.singh145@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(695014744698147),79fdecd3-c668-4569-8ec3-174a5fa3ab14,1111-133454-rb8bgmy7-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 17699, p25FileSize -> 15510, numDeletionVectorsRemoved -> 1, minFileSize -> 15510, numAddedFiles -> 1, maxFileSize -> 15510, p75FileSize -> 15510, p50FileSize -> 15510, numAddedBytes -> 15510)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-11T13:43:16.000Z,73729152433193,anupriya.singh145@gmail.com,DELETE,"Map(predicate -> [""(region#13001 = North)""])",null,List(695014744698147),79fdecd3-c668-4569-8ec3-174a5fa3ab14,1111-133454-rb8bgmy7-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2233, numDeletionVectorsUpdated -> 0, numDeletedRows -> 461, scanTimeMs -> 1346, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 857)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T13:40:36.000Z,73729152433193,anupriya.singh145@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""5b635da1-488e-4cf4-b614-6f622396b902"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-2073c408-9e87-4264-ba39-a9ea4e9e44f7"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-ab838920-dbd7-40d3-892e-cd258e0ae71a""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(695014744698147),65887895-08cc-43e3-916a-47139f217a8d,1111-133454-rb8bgmy7-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1859, numOutputBytes -> 17699)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
%sql
RESTORE TABLE workspace.retail.silver_sales TO VERSION AS OF 0;


table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
17699,1,1,1,15510,17699
